# 00 — Setup: get `selgate` running in this Kaggle notebook

This notebook (1) clones the `selective-gating` repo into `/kaggle/working`,
(2) makes the `selgate` package importable, (3) checks GPU + dataset wiring,
and (4) runs a 2-second model smoke test.

**Session settings needed** (right-hand Settings pane):
- **Accelerator:** GPU (T4 / P100 / T4x2)
- **Internet:** ON (needed for `git clone` and `pip install`)
- **Data:** attach your LOL-v1 dataset (Add data → search "LOL")

Run the cells **top to bottom**. If any cell fails, stop and fix before
running `01_train.ipynb`.

## 1. Clone the repo (private → authenticates with your `GITHUB_TOKEN` secret)

**One-time setup:** in this notebook's right-hand pane open **Secrets → Add
Secret**, name it `GITHUB_TOKEN`, value = a GitHub PAT with *Contents: Read &
write* on `pranjalkole/selective-gating`. Without the secret the clone is
anonymous and will fail (the repo is private).

In [ ]:
REPO_URL = "https://github.com/pranjalkole/selective-gating.git"   # private repo
BRANCH   = "master"

import os, subprocess, sys, shutil
from pathlib import Path

def _github_token():
    """Private repo: read GITHUB_TOKEN from a Kaggle secret (Add-ons -> Secrets)."""
    tok = os.environ.get("GITHUB_TOKEN")
    if tok:
        print("token source: env")
        return tok
    errors = []
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("token source: UserSecretsClient")
        return t
    except Exception as e:
        errors.append(f"UserSecretsClient -> {type(e).__name__}: {e}")
    try:
        from kaggle_secrets import userdata
        t = userdata.get("GITHUB_TOKEN")
        print("token source: userdata")
        return t
    except Exception as e:
        errors.append(f"userdata -> {type(e).__name__}: {e}")
    print("NO GITHUB_TOKEN SECRET AVAILABLE (repo is private, clone will fail):")
    for e in errors:
        print("  -", e)
    return None

token = _github_token()
print("clone auth:", "GITHUB_TOKEN secret" if token else "anonymous (public repo)")
clone_url = REPO_URL.replace("https://", f"https://oauth2:{token}@") if token else REPO_URL

repo = Path("/kaggle/working/selective-gating")
if repo.exists():
    if subprocess.run(["git", "-C", str(repo), "pull", clone_url, BRANCH]).returncode != 0:
        print("pull failed -> fresh clone")
        shutil.rmtree(repo)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url, str(repo)], check=True)
    # strip credentials from .git/config — kernel outputs are saved!
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)

print("repo at:", repo)

## 2. Install deps + make `selgate` importable (`/kaggle/input` is read-only, so everything runs from `/kaggle/working`)

In [ ]:
src = repo / "src"
os.environ["PYTHONPATH"] = f"{src}:{os.environ.get('PYTHONPATH', '')}"

# einops/pyyaml ship with the repo requirements; lpips+pyiqa are optional metric extras
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import selgate
print("selgate", selgate.__version__, "->", selgate.__file__)
assert str(repo / "src") in selgate.__file__, "selgate is not loaded from the cloned repo!"

## 3. Environment report

In [ ]:
import torch
from selgate.paths import is_kaggle, repo_root, find_lol_root, out_dir

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("on kaggle:", is_kaggle())
print("repo_root:", repo_root())

lol = find_lol_root()
print("LOL-v1 root:", lol)
if lol is None:
    raise FileNotFoundError(
        "LOL-v1 not found! Attach the dataset to this notebook (Add data), or set "
        "SELGATE_LOL_ROOT in the next cell to the folder that CONTAINS our485/ or Train/."
    )
from selgate.data import detect_layout
print("layout:", detect_layout(lol))

## 4. Model smoke test — builds Retinexformer + gate, one random forward pass

In [ ]:
import torch
from selgate.config import load_config
from selgate.models import build_backbone, count_params, GateNet, gate_param_count

cfg = load_config(repo / "configs/gate_pixel.yaml")
print("config:", cfg.get_path("name"), "| gate:", cfg.get_path("model.gate.granularity"))

model = build_backbone(**{k: cfg.get_path(f"model.backbone.{k}") for k in ("n_feat", "stage", "num_blocks")})
gate  = GateNet(granularity=cfg.get_path("model.gate.granularity"),
                 width=cfg.get_path("model.gate.width"),
                 depth=cfg.get_path("model.gate.depth"))

x = torch.rand(1, 3, 128, 128)
with torch.no_grad():
    g, _ = gate(x)
    y = model(x)

print(f"backbone params: {count_params(model):,}")
print(f"gate params:     {gate_param_count(gate):,}")
print("output:", tuple(y.shape), "| gate:", tuple(g.shape), f"| gate mean {g.mean():.3f}")
print("\nSETUP OK — next: run 01_train.ipynb")